# 05 · Reaction Kinetics — A Coupled ODE System

**Level:** Intermediate
**Prerequisites:** `01_intro_to_pinn`, `02_second_order_odes`

Chemical and process engineering runs on rate equations. Here we solve a simple
consecutive reaction

$$A \xrightarrow{k_1} B \xrightarrow{k_2} C$$

as a **coupled system of ODEs** with a single network that outputs all three
concentrations at once. This is the forward problem; notebook
`07_inverse_parameter_estimation` will *learn* the rate constants from data.

### What you will learn
1. A network with **multiple outputs** (one per species)
2. Writing a **system** of residuals
3. A conserved quantity (total mass) as a free accuracy check
4. Why non-dimensionalisation and loss weighting matter for stiff kinetics

### References
- Raissi, Perdikaris & Karniadakis (2019) — forward/inverse ODE-PDE framework
- Ji et al. (2021), *Stiff-PINN: Physics-Informed Neural Network for Stiff Chemical Kinetics*, J. Phys. Chem. A


## 1 · The kinetics

For $A\to B\to C$ with first-order rates:

$$\frac{dA}{dt}=-k_1 A,\qquad
\frac{dB}{dt}=k_1 A - k_2 B,\qquad
\frac{dC}{dt}=k_2 B,$$

with $A(0)=1,\;B(0)=C(0)=0$. Mass is conserved: $A+B+C=1$ for all $t$.

Analytical solution (for $k_1\neq k_2$):

$$A=e^{-k_1 t},\quad
B=\frac{k_1}{k_2-k_1}\left(e^{-k_1 t}-e^{-k_2 t}\right),\quad
C=1-A-B.$$


In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(0); np.random.seed(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

k1, k2 = 1.5, 0.6
T = 6.0

def exact(t):
    A = np.exp(-k1*t)
    B = k1/(k2-k1)*(np.exp(-k1*t)-np.exp(-k2*t))
    C = 1 - A - B
    return A, B, C


In [ ]:
class KineticsPINN(nn.Module):
    # one input (t) -> three outputs (A, B, C)
    def __init__(self, width=64, depth=4):
        super().__init__()
        layers = [1] + [width]*depth + [3]
        seq = []
        for i in range(len(layers)-1):
            seq.append(nn.Linear(layers[i], layers[i+1]))
            if i < len(layers)-2:
                seq.append(nn.Tanh())
        self.net = nn.Sequential(*seq)
        for m in self.net:
            if isinstance(m, nn.Linear):
                nn.init.xavier_normal_(m.weight); nn.init.zeros_(m.bias)
    def forward(self, t):
        return self.net(t)   # shape (N,3)

model = KineticsPINN().to(device)


In [ ]:
N_col = 500
t_col = torch.rand(N_col,1,device=device)*T; t_col.requires_grad_(True)
t0 = torch.zeros(1,1,device=device)
y0 = torch.tensor([[1.0, 0.0, 0.0]], device=device)

def d_dt(y, t):
    return torch.autograd.grad(y, t, torch.ones_like(y), create_graph=True)[0]

def loss_fn():
    y = model(t_col)                       # (N,3)
    A, B, C = y[:,0:1], y[:,1:2], y[:,2:3]
    At = d_dt(A, t_col); Bt = d_dt(B, t_col); Ct = d_dt(C, t_col)
    rA = At + k1*A
    rB = Bt - k1*A + k2*B
    rC = Ct - k2*B
    L_pde = torch.mean(rA**2 + rB**2 + rC**2)
    L_ic  = torch.mean((model(t0) - y0)**2)
    return L_pde + 20.0*L_ic


In [ ]:
opt = torch.optim.Adam(model.parameters(), lr=2e-3)
EPOCHS = 10000
hist = []
for ep in range(EPOCHS):
    opt.zero_grad(); loss = loss_fn(); loss.backward(); opt.step()
    hist.append(loss.item())
    if ep % 2000 == 0:
        print(f"epoch {ep:5d} | loss {loss.item():.3e}")
print("done")


In [ ]:
tt = torch.linspace(0, T, 300, device=device).view(-1,1)
with torch.no_grad():
    Y = model(tt).cpu().numpy()
tn = tt.cpu().numpy().ravel()
Ae, Be, Ce = exact(tn)

plt.figure(figsize=(8,5))
for i,(name,ye) in enumerate(zip("ABC",[Ae,Be,Ce])):
    plt.plot(tn, ye, 'k--', lw=1)
    plt.plot(tn, Y[:,i], lw=2, label=f'[{name}] PINN')
plt.plot([], [], 'k--', label='analytical')
plt.xlabel('time'); plt.ylabel('concentration'); plt.legend()
plt.title('Consecutive reaction A→B→C'); plt.tight_layout(); plt.show()

mass = Y.sum(axis=1)
print(f"mass conservation: min={mass.min():.4f} max={mass.max():.4f} (target 1.0)")
err = np.linalg.norm(Y - np.stack([Ae,Be,Ce],1))/np.linalg.norm(np.stack([Ae,Be,Ce],1))
print(f"relative L2 error (all species): {err:.3e}")


## 2 · Design notes

- **Shared trunk, multiple heads:** one network with a 3-D output learns all
  species jointly and lets them share features — usually better than three
  separate networks.
- **Mass conservation as a diagnostic:** we did *not* impose `A+B+C=1`. The fact
  that it comes out near 1 tells us the residuals are genuinely satisfied. You
  *could* add it as an extra soft constraint to accelerate training.
- **Stiffness:** if `k1` and `k2` differ by orders of magnitude, the system
  becomes *stiff* and plain PINNs struggle (fast transient + slow tail). See
  Ji et al. (2021) *Stiff-PINN* for the quasi-steady-state assumption trick.